# 16 — LLM Cluster Labelling

Assigns vivid, specific names to each cluster produced by notebooks 11
(embedding k-means) and 12 (values k-means).

**Prerequisites:** notebooks 11 and 12 must have been run first so that
`api/data/clusters/local_embedding_clusters.csv` and
`api/data/clusters/local_values_clusters.csv` exist.

For each LA × employment-group, all clusters in the group are sent to Gemini
in a single prompt so the model can give _contrastive_ names.
The updated `tribe_label` values are written back to the same API CSV files.


In [13]:
import sys, os, importlib, json, re, time
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

import data_pipeline.helpers.llm_prompts as _lp;  importlib.reload(_lp)
import data_pipeline.helpers.llm_gemini as _lgemini; importlib.reload(_lgemini)

# ── Config ────────────────────────────────────────────────────────────────────
LABEL_MODEL = "gemini-2.5-flash"
LABEL_DELAY = 5        # seconds between API calls

API_DIR = Path("../api/data/clusters")

TARGETS = [
    ("Embedding k-means", API_DIR / "local_embedding_clusters.csv"),
    ("Values k-means",    API_DIR / "local_values_clusters.csv"),
]

load_dotenv(dotenv_path=Path("..") / ".env")
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
raw_chat = _lgemini.make_gemini_caller(GOOGLE_API_KEY, LABEL_MODEL)

print(f"Model: {LABEL_MODEL}")
for name, path in TARGETS:
    exists = "✓" if path.exists() else "✗ MISSING"
    print(f"  {exists}  {name}: {path}")


Model: gemini-2.5-flash
  ✓  Embedding k-means: ../api/data/clusters/local_embedding_clusters.csv
  ✓  Values k-means: ../api/data/clusters/local_values_clusters.csv


In [14]:
def _extract_json(text: str) -> dict:
    clean = re.sub(r"```(?:json)?", "", text).strip(" `")
    return json.loads(clean)

def _is_default_label(label: str) -> bool:
    """True if the label is still a placeholder like 'Cluster 3'."""
    return isinstance(label, str) and re.fullmatch(r"Cluster \d+", label.strip()) is not None


for dataset_name, csv_path in TARGETS:
    if not csv_path.exists():
        print(f"\nSkipping {dataset_name} — file not found: {csv_path}")
        continue

    df_summary = pd.read_csv(csv_path)
    # Ensure tribe_description column exists
    if "tribe_description" not in df_summary.columns:
        df_summary["tribe_description"] = ""
    group_cols = ["ladcd", "group"] if "group" in df_summary.columns else ["ladcd"]
    n_batches  = df_summary.groupby(group_cols).ngroups

    print(f"\n{'='*60}")
    print(f"  {dataset_name}  ({n_batches} LA×group batches)")
    print(f"{'='*60}")

    for keys, grp in df_summary.groupby(group_cols):
        ladcd_val  = keys[0] if isinstance(keys, tuple) else keys
        group_val  = keys[1] if isinstance(keys, tuple) and len(keys) > 1 else None
        la_name    = grp["ladnm"].iloc[0]
        emp_label  = _lp.resolve_emp_label(group_val) if group_val is not None else None
        group_disp = emp_label or str(group_val) if group_val is not None else None

        # Skip if already labelled (resume support)
        if not all(_is_default_label(v) for v in grp["tribe_label"]):
            print(f"  {la_name} / {group_disp or 'all'}  — already labelled, skipping")
            continue

        print(f"  {la_name} / {group_disp or 'all'}  ({len(grp)} clusters) …",
              end=" ", flush=True)

        user_prompt = _lp.build_label_prompt(grp, group_disp or "", la_name, emp_label)
        response    = raw_chat(_lp.LABEL_SYSTEM_PROMPT, user_prompt, temperature=0.3)

        try:
            payload      = _extract_json(response)
            labels       = payload["labels"]
            descriptions = payload.get("descriptions", [""] * len(labels))
            if len(labels) != len(grp):
                raise ValueError(f"Got {len(labels)} labels for {len(grp)} clusters")
            if len(descriptions) != len(labels):
                descriptions = descriptions[:len(labels)] + [""] * (len(labels) - len(descriptions))
        except Exception as e:
            print(f"PARSE ERROR: {e}  — keeping default labels")
            time.sleep(LABEL_DELAY)
            continue

        for (idx, _row), label, desc in zip(grp.iterrows(), labels, descriptions):
            df_summary.at[idx, "tribe_label"]       = _lp.sanitise_name(label)
            df_summary.at[idx, "tribe_description"] = _lp.sanitise_name(desc)

        # Save after every batch so a timeout doesn't lose progress
        df_summary.to_csv(csv_path, index=False)
        print("OK  (saved)")
        time.sleep(LABEL_DELAY)

    print(f"\n  Finished {dataset_name} → {csv_path}")
    preview = ["ladcd", "ladnm"] + (["group"] if "group" in df_summary.columns else []) + ["cluster_id", "tribe_label", "tribe_description", "size"]
    print(df_summary[[c for c in preview if c in df_summary.columns]].to_string(index=False))

print("\nDone.")



  Embedding k-means  (36 LA×group batches)
  Blackpool / Employed  — already labelled, skipping
  Blackpool / Inactive  — already labelled, skipping
  Blackpool / On leave  — already labelled, skipping
  Blackpool / Retired  — already labelled, skipping
  Blackpool / Student / training  — already labelled, skipping
  Blackpool / Unemployed  — already labelled, skipping
  County Durham / Employed  — already labelled, skipping
  County Durham / Inactive  — already labelled, skipping
  County Durham / On leave  — already labelled, skipping
  County Durham / Retired  — already labelled, skipping
  County Durham / Student / training  — already labelled, skipping
  County Durham / Unemployed  — already labelled, skipping
  Hounslow / Employed  — already labelled, skipping
  Hounslow / Inactive  — already labelled, skipping
  Hounslow / On leave  — already labelled, skipping
  Hounslow / Retired  — already labelled, skipping
  Hounslow / Student / training  — already labelled, skipping
  Hou